In [38]:
# PFE Renault Tanger — Système d'alertes
## Notebook 05 : Alertes email automatiques + SHAP
### Objectif : envoyer un rapport quotidien intelligent par email

In [39]:
import pandas as pd
import numpy as np
import smtplib
import joblib
import shap
import matplotlib.pyplot as plt
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')

print("Librairies chargées ✓")

Librairies chargées ✓


In [40]:
import requests
import os
from datetime import datetime

DRIVE_FILE_ID = "1wR7e7dtbgBmnu5xDTB0kHpo9IpNsrUbp"  
FICHIER_LOCAL = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"

def telecharger_depuis_drive():
    """
    Télécharge un Google Sheets en format Excel (.xlsx)
    """
    # URL d'export Google Sheets → Excel
    url = f"https://docs.google.com/spreadsheets/d/{DRIVE_FILE_ID}/export?format=xlsx"
    
    print(f"[Drive] Téléchargement en cours...")
    print(f"[Drive] Heure : {datetime.now().strftime('%H:%M:%S')}")
    
    response = requests.get(url)
    
    if response.status_code == 200:
        with open(FICHIER_LOCAL, 'wb') as f:
            f.write(response.content)
        taille = os.path.getsize(FICHIER_LOCAL) / 1024
        print(f"[Drive] Fichier sauvegardé ✓")
        print(f"[Drive] Taille : {taille:.1f} Ko")
    else:
        print(f"[Drive] Erreur {response.status_code}")
        print("[Drive] Vérifie que le fichier est partagé publiquement")

telecharger_depuis_drive()

[Drive] Téléchargement en cours...
[Drive] Heure : 08:46:49
[Drive] Fichier sauvegardé ✓
[Drive] Taille : 739.1 Ko


In [41]:
import os
print("Dossier actuel :", os.getcwd())
print("Contenu de ../scripts :", os.listdir("../scripts"))

Dossier actuel : /Users/akrambelhaj/Desktop/PFE_Renault/notebooks
Contenu de ../scripts : ['.DS_Store', '__pycache__', 'pipeline_etl_eau_renault_1.py', '.ipynb_checkpoints']


In [42]:
import subprocess
import os  
import sys
print("[ETL] Mise à jour du dataset...")

# Supprimer l'ancien CSV pour forcer rechargement complet
if os.path.exists("../outputs/dataset_eau_propre.csv"):
    os.remove("../outputs/dataset_eau_propre.csv")
    print("[ETL] Ancien dataset supprimé ✓")

# Relancer le pipeline ETL
import sys
sys.path.append("../scripts")
import pipeline_etl_eau_renault_1 as etl

etl.FICHIER_EXCEL  = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"
etl.FICHIER_SORTIE = "../outputs/dataset_eau_propre.csv"
df_nouveau = etl.run_pipeline()

print(f"[ETL] Dataset mis à jour jusqu'au {df_nouveau['Date'].max().date()} ✓")

[ETL] Mise à jour du dataset...
[ETL] Ancien dataset supprimé ✓
 Pipeline ETL Eau - Renault Tanger
 Exécution : 2026-04-30 08:46:56
[ETL] Lecture du fichier : ../data/Synthèse_Eaux_2026_VF_(2).xlsm
[ETL] 117 lignes trouvées (de 2026-01-01 à 2026-04-27)
[ETL] Nettoyage des données...
  → ED_Total : 1 zéros remplacés par NaN
  → EOR_Total : 1 zéros remplacés par NaN
  → EP_Looker : 2 zéros remplacés par NaN
[ETL] Création des features...
  → 64 colonnes dans le dataset final
  → Jours normaux (TCM≥100) : 89
  → Jours arrêt                        : 28
  → Anomalies KPI détectées            : 15
  → Anomalies recyclage détectées      : 16
[ETL] Chargement du dataset...
[ETL] Fichier créé : ../outputs/dataset_eau_propre.csv (117 lignes)
 Pipeline terminé avec succès ✓
[ETL] Dataset mis à jour jusqu'au 2026-04-27 ✓


In [54]:

CONFIG_EMAIL = {
    'expediteur'     : 'akrambelhaj55@gmail.com',
    'mot_de_passe'   : 'ttro oydp uwtd vquo',   
    'destinataires'  : [
        'nisrineelmoubariki5@gmail.com',
        'akrambelhaj55@gmail.com'
    ],
    'smtp_serveur'   : 'smtp.gmail.com',
    'smtp_port'      : 587
}

SEUIL_OBJECTIF = 1.25   # m³/véhicule
SEUIL_ALERTE   = 1.25 * 1.15   # +15% = anomalie

print("Configuration chargée ✓")
print()
print("⚠️  IMPORTANT — Pour utiliser Gmail :")
print("   1. Va sur myaccount.google.com")
print("   2. Sécurité → Validation en 2 étapes → Active")
print("   3. Sécurité → Mots de passe des applications")
print("   4. Crée un mot de passe pour 'Mail'")
print("   5. Colle ce mot de passe dans CONFIG_EMAIL['mot_de_passe']")

Configuration chargée ✓

⚠️  IMPORTANT — Pour utiliser Gmail :
   1. Va sur myaccount.google.com
   2. Sécurité → Validation en 2 étapes → Active
   3. Sécurité → Mots de passe des applications
   4. Crée un mot de passe pour 'Mail'
   5. Colle ce mot de passe dans CONFIG_EMAIL['mot_de_passe']


In [55]:
df = pd.read_csv("../outputs/dataset_eau_propre.csv", parse_dates=['Date'])
df_prod = df[(df['TCM'] >= 100) & (df['is_weekend'] == 0)].copy().reset_index(drop=True)

model_xgb = joblib.load("../models/model_xgboost.pkl")

# ── CORRECTION Prophet : réentraîner avec données filtrées ──
from prophet import Prophet

df_prophet = df_prod[['Date','KPI_m3_veh','TCM']].copy()
df_prophet.columns = ['ds','y','TCM']
df_prophet = df_prophet.dropna(subset=['y'])

# Filtrer les outliers extrêmes (KPI > 3 = jours aberrants janvier)
df_prophet = df_prophet[df_prophet['y'] < 3.0].copy()

# Réentraîner Prophet sur données propres
model_prophet = Prophet(
    yearly_seasonality=False,   # pas assez de données pour annuelle
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive', # additif = pas de valeurs négatives
    changepoint_prior_scale=0.01 # très stable, peu de flexibilité
)
model_prophet.add_regressor('TCM')
model_prophet.fit(df_prophet)

# Recalculer SHAP
FEATURES = [
    'TCM', 'ED_Total', 'EOR_Total', 'EI_Facturation', 'EP_Facturation',
    'jour_semaine', 'mois', 'trimestre', 'is_lundi',
    'ratio_EOR_ED', 'taux_EP', 'taux_EI'
]
df_xgb  = df_prod[FEATURES + ['KPI_m3_veh', 'Date', 'E_Appro Facturation']].dropna().copy()
X       = df_xgb[FEATURES]

explainer   = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X)

# Sauvegarder le nouveau Prophet
joblib.dump(model_prophet, "../models/model_prophet.pkl")

print("Données et modèles chargés ✓")
print(f"Données Prophet filtrées : {len(df_prophet)} jours (KPI < 3.0)")
print(f"Dernier jour disponible  : {df_xgb['Date'].max().date()}")

08:48:22 - cmdstanpy - INFO - Chain [1] start processing
08:48:22 - cmdstanpy - INFO - Chain [1] done processing


Données et modèles chargés ✓
Données Prophet filtrées : 69 jours (KPI < 3.0)
Dernier jour disponible  : 2026-04-27


In [56]:
def calculer_kpis_dashboard(df_prod, model_prophet):
    """
    Calcule les 4 KPIs demandés par l'encadrant :
    - KPI YTD  : moyenne depuis le 1er Janvier jusqu'à aujourd'hui
    - KPI MTD  : moyenne depuis le 1er du mois en cours
    - KPI J-1  : KPI réel de l'avant-dernier jour saisi
    - KPI Année prédit : moyenne Prophet sur les jours restants 2026
    """
    from datetime import date
    import pandas as pd
    import numpy as np

    today      = pd.Timestamp(date.today())
    debut_ann  = pd.Timestamp(f"{today.year}-01-01")
    debut_mois = pd.Timestamp(f"{today.year}-{today.month:02d}-01")

    # Uniquement jours de production
    df_kpi = df_prod[df_prod['TCM'] >= 100].copy()
    df_kpi = df_kpi.sort_values('Date').reset_index(drop=True)

    # ── KPI YTD ─────────────────────────────────────────────
    df_ytd  = df_kpi[df_kpi['Date'] >= debut_ann]
    kpi_ytd = df_ytd['KPI_m3_veh'].mean()

    # ── KPI MTD ─────────────────────────────────────────────
    df_mtd  = df_kpi[df_kpi['Date'] >= debut_mois]
    kpi_mtd = df_mtd['KPI_m3_veh'].mean()

    # ── KPI J-1 : lu directement depuis Excel ───────────────────
    df_j1 = pd.read_excel(
        "../data/Synthèse_Eaux_2026_VF_(2).xlsm",
        sheet_name="database_Eaux",
        header=0
    )
    df_j1.columns = df_j1.columns.str.strip()
    df_j1 = df_j1[df_j1['TCM'] >= 100].dropna(subset=['TCM'])
    df_j1 = df_j1.sort_values('Date').reset_index(drop=True)

    # Colonne correcte confirmée par encadrant
    COL_APPRO = 'E_Appro Facturation'

    derniere = df_j1.iloc[-1]
    kpi_j1   = derniere[COL_APPRO] / derniere['TCM']
    date_j1  = derniere['Date'].date()
    tcm_j1   = derniere['TCM']
    conso_j1 = derniere[COL_APPRO]

    # ── KPI PRÉDIT ANNÉE ────────────────────────────────────
    fin_annee   = pd.Timestamp(f"{today.year}-12-31")
    jours_rest  = pd.bdate_range(start=today + pd.Timedelta(days=1), end=fin_annee)

    tcm_moyen   = df_prod['TCM'].mean()

    df_prophet_base = df_prod[['Date','KPI_m3_veh','TCM']].copy()
    df_prophet_base.columns = ['ds','y','TCM']
    df_prophet_base = df_prophet_base.dropna(subset=['y'])
    df_prophet_base = df_prophet_base[df_prophet_base['y'] < 3.0]

    df_future_ann = pd.concat([
        df_prophet_base[['ds','TCM']],
        pd.DataFrame({'ds': jours_rest, 'TCM': tcm_moyen})
    ], ignore_index=True)

    forecast_ann    = model_prophet.predict(df_future_ann)
    pred_restants   = forecast_ann[forecast_ann['ds'].isin(jours_rest)]['yhat']
    pred_restants   = pred_restants.clip(lower=0.5, upper=2.0) 
    kpi_predit_ann  = pred_restants.mean()

    # ── KPI MOYEN PRÉVU ANNUEL (historique + futur) ─────────
    kpi_reel_jours  = df_ytd['KPI_m3_veh'].mean()
    nb_jours_reel   = len(df_ytd)
    nb_jours_futur  = len(pred_restants)
    
    poids_reel  = nb_jours_reel
    poids_futur = nb_jours_futur * 0.3  

    kpi_annee_complet = (
        (kpi_reel_jours * poids_reel + kpi_predit_ann * poids_futur) 
        / (poids_reel + poids_futur)
    )

    # ── STATUT PAR RAPPORT À L'OBJECTIF ─────────────────────
    OBJECTIF = 1.25

    def statut_kpi(val):
        if val > OBJECTIF * 1.15:
            return {"emoji":"🔴", "texte":"Au-dessus de l'objectif", "couleur":"#A32D2D", "bg":"#FCEBEB"}
        elif val > OBJECTIF:
            return {"emoji":"🟡", "texte":"Proche de l'objectif", "couleur":"#854F0B", "bg":"#FAEEDA"}
        else:
            return {"emoji":"🟢", "texte":"Sous l'objectif ✓", "couleur":"#27500A", "bg":"#EAF3DE"}

    result = {
        'kpi_ytd': kpi_ytd,
        'kpi_mtd': kpi_mtd,
        'kpi_j1': kpi_j1,
        'date_j1': date_j1,
        'tcm_j1': tcm_j1,
        'conso_j1': conso_j1,
        'kpi_predit_annee': kpi_annee_complet,
        'nb_jours_reel': nb_jours_reel,
        'nb_jours_futur': nb_jours_futur,
        'objectif': OBJECTIF,
        'statut_ytd': statut_kpi(kpi_ytd),
        'statut_mtd': statut_kpi(kpi_mtd),
        'statut_j1': statut_kpi(kpi_j1),
        'statut_annee': statut_kpi(kpi_annee_complet),
    }

    print("=== KPIs DASHBOARD ===")
    print(f"KPI YTD            : {kpi_ytd:.3f} m³/véh  {statut_kpi(kpi_ytd)['emoji']}")
    print(f"KPI MTD            : {kpi_mtd:.3f} m³/véh  {statut_kpi(kpi_mtd)['emoji']}")
    print(f"KPI J-1 ({date_j1}) : {kpi_j1:.3f} m³/véh  {statut_kpi(kpi_j1)['emoji']}")
    print(f"KPI Prédit année   : {kpi_annee_complet:.3f} m³/véh  {statut_kpi(kpi_annee_complet)['emoji']}")
    print(f"Objectif 2026      : {OBJECTIF} m³/véh")

    return result

# Exécution
kpis = calculer_kpis_dashboard(df_prod, model_prophet)

=== KPIs DASHBOARD ===
KPI YTD            : 1.005 m³/véh  🟢
KPI MTD            : 1.077 m³/véh  🟢
KPI J-1 (2026-04-27) : 0.776 m³/véh  🟢
KPI Prédit année   : 1.357 m³/véh  🟡
Objectif 2026      : 1.25 m³/véh


In [57]:
def analyser_jour_courant(df_xgb, shap_values, model_xgb):
    """
    Analyse le dernier jour disponible dans le dataset.
    Retourne un dictionnaire avec toutes les infos du rapport.
    """
    idx      = len(df_xgb) - 1
    row      = df_xgb.iloc[idx]
    date_j   = row['Date'].date()
    kpi_reel = row['KPI_m3_veh']
    kpi_pred = model_xgb.predict(X.iloc[[idx]])[0]

    # Statut
    if kpi_reel > SEUIL_ALERTE:
        statut       = "ANOMALIE"
        statut_emoji = "🔴"
        couleur      = "#FCEBEB"
    elif kpi_reel > SEUIL_OBJECTIF:
        statut       = "ATTENTION"
        statut_emoji = "🟡"
        couleur      = "#FAEEDA"
    else:
        statut       = "NORMAL"
        statut_emoji = "🟢"
        couleur      = "#EAF3DE"

    # Top 3 causes SHAP — uniquement features métier
    contribs = pd.Series(shap_values[idx], index=FEATURES)

    # ── CORRECTION : exclure les features non parlantes ──────
    FEATURES_EXCLURE = [
        'jour_semaine', 'mois', 'trimestre',
        'is_lundi', 'taux_EP', 'taux_EI'
    ]
    contribs_metier = contribs.drop(
        [f for f in FEATURES_EXCLURE if f in contribs.index]
    )
    top3 = contribs_metier.abs().nlargest(3)
    # ─────────────────────────────────────────────────────────

    noms_lisibles = {
        'TCM'          : 'Production (TCM)',
        'ED_Total'     : 'Eau déminéralisée',
        'EOR_Total'    : 'Eau osmosée recyclée',
        'EI_Facturation': 'Eau industrielle (facturation)',
        'EP_Facturation': 'Eau potable (facturation)',
        'ratio_EOR_ED' : 'Taux recyclage EOR/ED',
    }

    causes = []
    for feat, _ in top3.items():
        val_shap   = contribs[feat]
        val_reelle = X.iloc[idx][feat]
        nom        = noms_lisibles.get(feat, feat)
        direction  = "↑ augmente" if val_shap > 0 else "↓ réduit"
        causes.append({
            'feature'  : nom,
            'valeur'   : val_reelle,
            'shap'     : val_shap,
            'direction': direction
        })

    return {
        'date'        : date_j,
        'kpi_reel'    : kpi_reel,
        'kpi_pred'    : kpi_pred,
        'tcm'         : row['TCM'],
        'conso'       : row['E_Appro Facturation'],
        'statut'      : statut,
        'statut_emoji': statut_emoji,
        'couleur'     : couleur,
        'causes'      : causes
    }

analyse = analyser_jour_courant(df_xgb, shap_values, model_xgb)
print(f"Date analysée  : {analyse['date']}")
print(f"Statut         : {analyse['statut_emoji']} {analyse['statut']}")
print(f"KPI réel       : {analyse['kpi_reel']:.3f} m³/véh")
print(f"KPI prédit     : {analyse['kpi_pred']:.3f} m³/véh")
print(f"TCM            : {analyse['tcm']:.0f} véhicules")
print()
print("Top 3 causes SHAP (features métier uniquement) :")
for c in analyse['causes']:
    print(f"  {c['direction']} le KPI — {c['feature']} = {c['valeur']:.1f}  (SHAP={c['shap']:+.3f})")

Date analysée  : 2026-04-27
Statut         : 🟢 NORMAL
KPI réel       : 0.694 m³/véh
KPI prédit     : 0.889 m³/véh
TCM            : 1375 véhicules

Top 3 causes SHAP (features métier uniquement) :
  ↓ réduit le KPI — Production (TCM) = 1375.0  (SHAP=-0.112)
  ↑ augmente le KPI — Eau potable (facturation) = 348.0  (SHAP=+0.054)
  ↓ réduit le KPI — Eau industrielle (facturation) = 694.0  (SHAP=-0.047)


In [58]:
def predire_7_jours(model_prophet, df_prod):
    """
    Génère les prédictions à partir d'AUJOURD'HUI
    et non depuis la dernière date du dataset.
    """
    from datetime import date
    import pandas as pd

    tcm_moyen = df_prod['TCM'].mean()

    df_prophet = df_prod[['Date','KPI_m3_veh','TCM']].copy()
    df_prophet.columns = ['ds','y','TCM']
    df_prophet = df_prophet.dropna(subset=['y'])

    # ── CORRECTION : calculer combien de jours ouvrés
    # manquent entre la dernière date et aujourd'hui + 7 jours
    derniere_date = df_prophet['ds'].max()
    aujourd_hui   = pd.Timestamp(date.today())

    # Générer tous les jours ouvrés depuis la dernière date jusqu'à +7j futurs
    toutes_dates = pd.bdate_range(
        start=derniere_date + pd.Timedelta(days=1),
        end=aujourd_hui + pd.Timedelta(days=10)  # marge suffisante
    )

    # Garder uniquement les 7 prochains jours ouvrés FUTURS
    jours_futurs = pd.DataFrame({'ds': toutes_dates})
    jours_futurs = jours_futurs[jours_futurs['ds'] > aujourd_hui].head(7)

    # Construire le dataframe future complet pour Prophet
    df_future = pd.concat([
        df_prophet[['ds','TCM']],
        jours_futurs.assign(TCM=tcm_moyen)
    ], ignore_index=True)

    # Prédire
    forecast = model_prophet.predict(df_future)

    # Garder uniquement les 7 jours futurs
    predictions = forecast[forecast['ds'].isin(jours_futurs['ds'])][
        ['ds','yhat','yhat_lower','yhat_upper']
    ].copy()
    predictions.columns = ['date','kpi_predit','borne_basse','borne_haute']
    predictions['statut'] = predictions['kpi_predit'].apply(
        lambda x: '🔴 Alerte'    if x > SEUIL_ALERTE
                  else ('🟡 Attention' if x > SEUIL_OBJECTIF
                  else '🟢 Normal')
    )

    return predictions

predictions_7j = predire_7_jours(model_prophet, df_prod)

print("=== PRÉDICTIONS 7 PROCHAINS JOURS RÉELS ===")
print(predictions_7j.round(3).to_string(index=False))

=== PRÉDICTIONS 7 PROCHAINS JOURS RÉELS ===
      date  kpi_predit  borne_basse  borne_haute      statut
2026-05-01       1.273        1.071        1.495 🟡 Attention
2026-05-04       1.284        1.083        1.506 🟡 Attention
2026-05-05       1.333        1.136        1.553 🟡 Attention
2026-05-06       1.416        1.196        1.623 🟡 Attention
2026-05-07       1.372        1.171        1.582 🟡 Attention
2026-05-08       1.310        1.107        1.520 🟡 Attention


In [59]:
def barre_kpi(valeur, objectif=1.25, max_val=2.0):
    """Génère une mini barre de progression HTML."""
    pct = min(int((valeur / max_val) * 100), 100)
    pct_obj = min(int((objectif / max_val) * 100), 100)
    couleur = "#A32D2D" if valeur > objectif * 1.15 else \
              "#854F0B" if valeur > objectif else "#27500A"
    return f"""
    <div style="margin-top:5px; background:#eee; border-radius:4px; height:7px; position:relative;">
      <div style="width:{pct}%; background:{couleur}; height:7px; border-radius:4px;"></div>
      <div style="position:absolute; left:{pct_obj}%; top:-2px; width:2px; height:11px; background:orange;"></div>
    </div>
    <div style="font-size:10px; color:#aaa; margin-top:2px;">▲ Objectif 1.25</div>
    """

def construire_email_html(analyse, predictions_7j, kpis):
    # ── BLOC SHAP ─────────────────────────────────────────────
    lignes_shap = ""
    for c in analyse['causes']:
        couleur_shap = "#A32D2D" if c['shap'] > 0 else "#27500A"
        lignes_shap += f"""
        <tr>
            <td style="padding:7px 12px;">{c['feature']}</td>
            <td style="padding:7px 12px; text-align:center;">{c['valeur']:.1f}</td>
            <td style="padding:7px 12px; text-align:center; color:{couleur_shap}; font-weight:500;">
                {c['shap']:+.3f}
            </td>
            <td style="padding:7px 12px;">{c['direction']} le KPI</td>
        </tr>"""

    # ── BLOC PRÉVISIONS 7 JOURS ───────────────────────────────
    lignes_prev = ""
    jours_fr = {'Monday':'Lundi','Tuesday':'Mardi','Wednesday':'Mercredi','Thursday':'Jeudi','Friday':'Vendredi','Saturday':'Samedi','Sunday':'Dimanche'}
    mois_fr = {1:'Jan',2:'Fév',3:'Mar',4:'Avr',5:'Mai',6:'Jun',7:'Jul',8:'Aoû',9:'Sep',10:'Oct',11:'Nov',12:'Déc'}

    for _, row in predictions_7j.iterrows():
        if "Alerte" in row['statut']:
            bg_row, bg_kpi, icone = "#FCEBEB", "#A32D2D", "🔴"
            msg = "Consommation prévue trop élevée"
        elif "Attention" in row['statut']:
            bg_row, bg_kpi, icone = "#FAEEDA", "#854F0B", "🟡"
            msg = "Proche du seuil — à surveiller"
        else:
            bg_row, bg_kpi, icone = "#EAF3DE", "#27500A", "🟢"
            msg = "Consommation prévue normale"

        nom_jour = jours_fr.get(row['date'].strftime('%A'), row['date'].strftime('%A'))
        date_str = f"{nom_jour} {row['date'].strftime('%d')} {mois_fr.get(row['date'].month, '')}"
        
        kpi_val = max(0, row['kpi_predit'])
        pct = min(int((kpi_val / 2.0) * 100), 100)
        pct_obj = min(int((1.25 / 2.0) * 100), 100)

        lignes_prev += f"""
        <tr style="border-bottom:1px solid #e0e0e0;">
          <td style="padding:12px 14px; background:{bg_row}; border-left:4px solid {bg_kpi}; width:130px;">
            <div style="font-weight:600; font-size:13px; color:#222;">{date_str}</div>
          </td>
          <td style="padding:12px 14px; background:white; width:200px;">
            <div style="font-size:18px; font-weight:700; color:{bg_kpi};">{kpi_val:.2f} <span style="font-size:12px; font-weight:400; color:#666;">m³/véh</span></div>
            <div style="margin-top:6px; background:#eee; border-radius:4px; height:8px; position:relative;">
              <div style="width:{pct}%; background:{bg_kpi}; height:8px; border-radius:4px;"></div>
              <div style="position:absolute; left:{pct_obj}%; top:-3px; width:2px; height:14px; background:orange;"></div>
            </div>
          </td>
          <td style="padding:12px 14px; background:white; font-size:13px;">{icone} {msg}</td>
        </tr>"""

    # ── CONSTRUCTION DU DASHBOARD KPI ─────────────────────────
    bloc_kpi = f"""
    <div style="padding:16px 24px; background:#f9f9f9; margin-top:2px;">
      <h3 style="margin:0 0 14px; font-size:14px;">📊 Tableau de bord KPI — Consommation eau</h3>
      <table style="width:100%; border-collapse:collapse;">
        <tr>
          <td style="padding:0 6px 0 0; width:25%; vertical-align:top;">
            <div style="background:white; border-radius:8px; padding:12px 14px; border-top:3px solid {kpis['statut_ytd']['couleur']};">
              <div style="font-size:11px; color:#888;">KPI YTD</div>
              <div style="font-size:22px; font-weight:700; color:{kpis['statut_ytd']['couleur']};">{kpis['kpi_ytd']:.3f}</div>
              {barre_kpi(kpis['kpi_ytd'])}
            </div>
          </td>
          <td style="padding:0 6px; width:25%; vertical-align:top;">
            <div style="background:white; border-radius:8px; padding:12px 14px; border-top:3px solid {kpis['statut_mtd']['couleur']};">
              <div style="font-size:11px; color:#888;">KPI MTD</div>
              <div style="font-size:22px; font-weight:700; color:{kpis['statut_mtd']['couleur']};">{kpis['kpi_mtd']:.3f}</div>
              {barre_kpi(kpis['kpi_mtd'])}
            </div>
          </td>
          <td style="padding:0 6px; width:25%; vertical-align:top;">
            <div style="background:white; border-radius:8px; padding:12px 14px; border-top:3px solid {kpis['statut_j1']['couleur']};">
              <div style="font-size:11px; color:#888;">Réel J-1</div>
              <div style="font-size:22px; font-weight:700; color:{kpis['statut_j1']['couleur']};">{kpis['kpi_j1']:.3f}</div>
              {barre_kpi(kpis['kpi_j1'])}
            </div>
          </td>
          <td style="padding:0 0 0 6px; width:25%; vertical-align:top;">
            <div style="background:white; border-radius:8px; padding:12px 14px; border-top:3px solid {kpis['statut_annee']['couleur']};">
              <div style="font-size:11px; color:#888;">Prédit 2026</div>
              <div style="font-size:22px; font-weight:700; color:{kpis['statut_annee']['couleur']};">{kpis['kpi_predit_annee']:.3f}</div>
              {barre_kpi(kpis['kpi_predit_annee'])}
            </div>
          </td>
        </tr>
      </table>
    </div>"""

    # ── ASSEMBLAGE FINAL ──────────────────────────────────────
    html = f"""
    <html>
    <body style="font-family:Arial,sans-serif; max-width:700px; margin:auto; color:#222;">
      <div style="background:#1a1a2e; padding:20px 24px; border-radius:8px 8px 0 0;">
        <h2 style="color:white; margin:0;">🌊 Rapport Eau Quotidien</h2>
        <p style="color:#aaa; margin:4px 0 0;">Renault Tanger — {analyse['date'].strftime('%d %B %Y')}</p>
      </div>

      {bloc_kpi}

      <div style="padding:16px 24px; background:#f9f9f9; margin-top:2px;">
        <h3 style="margin:0 0 12px;">🧠 Explication IA (SHAP)</h3>
        <table style="width:100%; border-collapse:collapse; font-size:13px; background:white;">
          <thead><tr style="background:#eee;"><th style="padding:8px;">Variable</th><th>Valeur</th><th>Impact</th><th>Effet</th></tr></thead>
          <tbody>{lignes_shap}</tbody>
        </table>
      </div>

      <div style="padding:16px 24px; margin-top:2px;">
        <h3 style="margin:0 0 12px;">📅 Prévisions 7 jours</h3>
        <table style="width:100%; border-collapse:collapse; box-shadow:0 1px 4px rgba(0,0,0,0.08);">
          <tbody>{lignes_prev}</tbody>
        </table>
      </div>

      <div style="background:#f0f0f0; padding:12px 24px; border-radius:0 0 8px 8px; font-size:11px; color:#888;">
        <p>Rapport généré automatiquement — Système IA Eau | PFE 2026</p>
      </div>
    </body>
    </html>
    """
    return html

# Exécution
email_html = construire_email_html(analyse, predictions_7j, kpis)
print("Email HTML construit ✓")
print(f"Taille : {len(email_html)} caractères")

Email HTML construit ✓
Taille : 11720 caractères


In [60]:
def envoyer_email(html, analyse, config):
    """
    Envoie l'email via Gmail SMTP.
    """
    sujet = (
        f"[🔴 ALERTE EAU] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
        if analyse['statut'] == 'ANOMALIE'
        else f"[🟢 Rapport Eau] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
    )

    msg = MIMEMultipart('alternative')
    msg['Subject'] = sujet
    msg['From']    = config['expediteur']        # ← correction 1 : config pas CONFIG_EMAIL
    msg['To']      = ', '.join(config['destinataires'])
    msg.attach(MIMEText(html, 'html'))

    try:
        with smtplib.SMTP(config['smtp_serveur'], config['smtp_port']) as serveur:
            serveur.starttls()
            serveur.login(config['expediteur'], config['mot_de_passe'])
            serveur.sendmail(
                config['expediteur'],
                config['destinataires'],
                msg.as_string()              # ← correction 2 : msg.as_string() pas [msg.as](http://...)
            )
        print(f"✅ Email envoyé avec succès !")
        print(f"   Sujet : {sujet}")
        print(f"   À     : {config['destinataires']}")
    except Exception as e:
        print(f"❌ Erreur envoi : {e}")
        print("   Vérifie le mot de passe application Gmail")

# Test d'envoi — décommente quand prêt
#envoyer_email(email_html, analyse, CONFIG_EMAIL)
print("Fonction prête ✓")

Fonction prête ✓


In [61]:
def envoyer_email(html, analyse, CONFIG_EMAIL):
    """
    Envoie l'email via Gmail SMTP.
    """
    sujet = (
        f"[🔴 ALERTE EAU] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
        if analyse['statut'] == 'ANOMALIE'
        else f"[🟢 Rapport Eau] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
    )

    msg = MIMEMultipart('alternative')
    msg['Subject'] = sujet
    msg['From']    = config['expediteur']
    msg['To']      = ', '.join(config['destinataires'])
    msg.attach(MIMEText(html, 'html'))

    try:
        with smtplib.SMTP(config['smtp_serveur'], config['smtp_port']) as serveur:
            serveur.starttls()
            serveur.login(config['expediteur'], config['mot_de_passe'])
            serveur.sendmail(
                config['expediteur'],
                config['destinataires'],
                msg.as_string()
            )
        print(f"✅ Email envoyé avec succès !")
        print(f"   Sujet : {sujet}")
        print(f"   À     : {config['destinataires']}")
    except Exception as e:
        print(f"❌ Erreur envoi : {e}")
        print("   Vérifie le mot de passe application Gmail")

# Test d'envoi — décommente quand prêt
# envoyer_email(email_html, analyse, CONFIG_EMAIL)
print("Fonction prête ✓")
print("→ Décommente la dernière ligne pour envoyer l'email")

Fonction prête ✓
→ Décommente la dernière ligne pour envoyer l'email


In [62]:
def envoyer_email(html, analyse, config):
    """
    Envoie l'email via Gmail SMTP.
    """
    sujet = (
        f"[🔴 ALERTE EAU] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
        if analyse['statut'] == 'ANOMALIE'
        else f"[🟢 Rapport Eau] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
    )

    msg = MIMEMultipart('alternative')
    msg['Subject'] = sujet
    msg['From']    = config['expediteur']
    msg['To']      = ', '.join(config['destinataires'])
    msg.attach(MIMEText(html, 'html'))

    try:
        with smtplib.SMTP(config['smtp_serveur'], config['smtp_port']) as serveur:
            serveur.starttls()
            serveur.login(config['expediteur'], config['mot_de_passe'])
            serveur.sendmail(
                config['expediteur'],
                config['destinataires'],
                msg.as_string()
            )
        print(f"✅ Email envoyé avec succès !")
        print(f"   Sujet : {sujet}")
        print(f"   À     : {config['destinataires']}")
    except Exception as e:
        print(f"❌ Erreur envoi : {e}")
        print("   Vérifie le mot de passe application Gmail")

# Test d'envoi — décommente quand prêt
# envoyer_email(email_html, analyse, CONFIG_EMAIL)
print("Fonction prête ✓")
print("→ Décommente la dernière ligne pour envoyer l'email")

Fonction prête ✓
→ Décommente la dernière ligne pour envoyer l'email


In [63]:
# Sauvegarder l'email en HTML pour le prévisualiser dans le navigateur
with open("../outputs/email_rapport_eau.html", "w", encoding="utf-8") as f:
    f.write(email_html)

print("Email sauvegardé → ../outputs/email_rapport_eau.html")
print()
print("Pour prévisualiser :")
print("  Double-clique sur le fichier email_rapport_eau.html")
print("  Il s'ouvre dans ton navigateur exactement comme il sera reçu")

Email sauvegardé → ../outputs/email_rapport_eau.html

Pour prévisualiser :
  Double-clique sur le fichier email_rapport_eau.html
  Il s'ouvre dans ton navigateur exactement comme il sera reçu


In [64]:
def rapport_quotidien_complet():
    """
    Fonction principale appelée chaque matin par le cron.
    Regroupe tout : analyse + prédictions + email.
    """
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Démarrage rapport quotidien...")

    # 1. Analyser le jour courant
    analyse = analyser_jour_courant(df_xgb, shap_values, model_xgb)
    print(f"  Statut : {analyse['statut_emoji']} {analyse['statut']}")

    # 2. Prédictions 7 jours
    predictions = predire_7_jours(model_prophet, df_prod)
    print(f"  Prédictions 7j générées ✓")

    # 3. Construire email
    html = construire_email_html(analyse, predictions,kpis)
    print(f"  Email HTML construit ✓")

    # 4. Envoyer seulement si anomalie OU heure = 8h
    heure_actuelle = datetime.now().hour
    if analyse['statut'] == 'ANOMALIE':
        print("  ⚠️  Anomalie détectée → envoi immédiat")
        envoyer_email(html, analyse, CONFIG_EMAIL)
    elif heure_actuelle == 8:
        print("  📧 Rapport matinal → envoi")
        envoyer_email(html, analyse, CONFIG_EMAIL)
    else:
        print("  ℹ️  Pas d'anomalie + heure != 8h → email non envoyé")
        print("      (sauvegardé dans outputs/email_rapport_eau.html)")

    return analyse, predictions

# Lancer le rapport
analyse_finale, pred_finale = rapport_quotidien_complet()

[08:48:31] Démarrage rapport quotidien...
  Statut : 🟢 NORMAL
  Prédictions 7j générées ✓
  Email HTML construit ✓
  📧 Rapport matinal → envoi
✅ Email envoyé avec succès !
   Sujet : [🟢 Rapport Eau] 2026-04-27 — KPI=0.694 m³/véh
   À     : ['nisrineelmoubariki5@gmail.com', 'akrambelhaj55@gmail.com']
